In [2]:
import pandas as pd
import re

print("=" * 80)
print("Diagnosing Missing Queries in ASR Analysis")
print("=" * 80)
print()

# ============================================================================
# 第一步：读取数据
# ============================================================================

# 读取 logs
try:
    logs_df = pd.read_csv('logs.csv')
    print(f"✓ Logs loaded: {len(logs_df)} records")
except:
    try:
        # 从 SQL 解析
        def parse_sql_insert_statements(sql_file_path, table_name='logs'):
            print(f"Parsing SQL file: {sql_file_path}")
            data = []
            with open(sql_file_path, 'r', encoding='utf-8') as f:
                buffer = ""
                insert_pattern = f"INSERT INTO `{table_name}` VALUES "
                for line_num, line in enumerate(f, 1):
                    if line_num % 100000 == 0:
                        print(f"  Processed {line_num} lines...")
                    if line.startswith('--') or line.startswith('/*') or line.startswith('*/') or not line.strip():
                        continue
                    buffer += line
                    if insert_pattern in buffer:
                        try:
                            values_start = buffer.find(insert_pattern) + len(insert_pattern)
                            values_section = buffer[values_start:]
                            pattern = r'\(([^)]+)\)'
                            matches = re.findall(pattern, values_section)
                            for match in matches:
                                values = []
                                current = ""
                                in_quotes = False
                                quote_char = None
                                for char in match:
                                    if char in ('"', "'") and (not in_quotes or char == quote_char):
                                        if in_quotes and char == quote_char:
                                            in_quotes = False
                                            quote_char = None
                                        elif not in_quotes:
                                            in_quotes = True
                                            quote_char = char
                                        current += char
                                    elif char == ',' and not in_quotes:
                                        values.append(current.strip().strip('"').strip("'"))
                                        current = ""
                                    else:
                                        current += char
                                if current:
                                    values.append(current.strip().strip('"').strip("'"))
                                if len(values) == 10:
                                    data.append(values)
                            buffer = ""
                        except Exception as e:
                            buffer = ""
                            continue
            columns = ['id', 'user_id', 'qid', 'docno', 'event_type', 
                       'start_idx', 'end_idx', 'duration', 'pass_flag', 'timestamp']
            df = pd.DataFrame(data, columns=columns)
            df['id'] = pd.to_numeric(df['id'], errors='coerce')
            df['qid'] = pd.to_numeric(df['qid'], errors='coerce')
            df['start_idx'] = pd.to_numeric(df['start_idx'], errors='coerce')
            df['end_idx'] = pd.to_numeric(df['end_idx'], errors='coerce')
            df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
            df['pass_flag'] = pd.to_numeric(df['pass_flag'], errors='coerce')
            return df
        
        logs_df = parse_sql_insert_statements('backup_qe_first_half.sql')
        print(f"✓ Logs parsed from SQL: {len(logs_df)} records")
    except:
        print("✗ Cannot load logs data")
        exit(1)

# 读取 interleaving_results
try:
    interleaving_df = pd.read_csv('result_interleaved_with_text.csv')
    print(f"✓ Interleaving results loaded: {len(interleaving_df)} records")
except:
    print("✗ interleaving_results.csv not found")
    exit(1)

print()

# ============================================================================
# 第二步：筛选目标用户（硬编码27个用户）
# ============================================================================

target_users = ['5b68c9eb87af310001584803',
                '55d51a6b8ce09000127d4821',
                '67181a55422d48f5727f08ec',
                '595cd88562431800019d691a',
                '610c66163e803a3564dfaf9c',
                '66435e2e5c5b5aceb2195908',
                '628f572a2d1304a34b9e1a37',
                '65de30b34b701c678a13c75b',
                '5f540fc665e3b6148c04f10e',
                '5f1088e27a43fe0c64ca5814',
                '66cc6eb364154a51d4d44eb1',
                '66cec5a3fdf1fe2c010e9971',
                '5c6e60b04e08ad00018cc995',
                '673e222bdb1a35935b3075f1',
                '659c067de92634488b349626',
                '5d68c8aa40524c00189e8ac2',
                '66fa1820329fa9bb88f2e3c6',
                '6012e262527c380ccb2a6c74',
                '668a9e53e7690b3b64d48e92',
                '5ca102f57114700016da340d',
                '5c6709ab926a5b0001eb29c1',
                '66d9cd06e207a93377f7820e',
                '6151e479b4ee16668948c7f2',
                '5d542c3a5af5900019bc80c4',
                '5f565a2b33c152071057bb04',
                '62d6d77df59450e545fb22f6',
                '63cedb764240865b7177c745']

print("Filtering for target users...")
logs_df = logs_df[logs_df['user_id'].isin(target_users)].copy()
print(f"✓ Filtered logs: {len(logs_df)} records from {logs_df['user_id'].nunique()} users")
print()

# 时间筛选（可选）
EXPERIMENT_START_DATE = '2025-11-01'  # 修改为你的实验开始日期

if EXPERIMENT_START_DATE is not None:
    logs_df['timestamp'] = pd.to_datetime(logs_df['timestamp'], errors='coerce')
    experiment_start = pd.to_datetime(EXPERIMENT_START_DATE)
    before_time_filter = len(logs_df)
    logs_df = logs_df[logs_df['timestamp'] >= experiment_start].copy()
    after_time_filter = len(logs_df)
    print(f"✓ Time filtered: removed {before_time_filter - after_time_filter} old records")
    print()

# ============================================================================
# 第三步：创建文档标签映射
# ============================================================================

doc_label_map = {}
for _, row in interleaving_df.iterrows():
    key = (str(row['qid']), str(row['docno']))
    doc_label_map[key] = row['origin_label']

print(f"Document label mapping created: {len(doc_label_map)} documents")
print()

# ============================================================================
# 第四步：分析 PASSAGE_SELECTION 事件
# ============================================================================

passage_logs = logs_df[logs_df['event_type'] == 'PASSAGE_SELECTION'].copy()
print(f"Total PASSAGE_SELECTION events: {len(passage_logs)}")
print()

# 添加 origin_label
def get_origin_label(row):
    qid_str = str(int(row['qid'])) if pd.notna(row['qid']) else '0'
    docno_str = str(row['docno'])
    key = (qid_str, docno_str)
    return doc_label_map.get(key, 'Unknown')

passage_logs['origin_label'] = passage_logs.apply(get_origin_label, axis=1)

# ============================================================================
# 第五步：分析所有查询
# ============================================================================

print("=" * 80)
print("Query Analysis")
print("=" * 80)
print()

all_queries = passage_logs['qid'].unique()
print(f"Total unique queries in logs: {len(all_queries)}")
print()

# 分析每个查询的文档分布
query_analysis = []

for qid in sorted(all_queries):
    query_passages = passage_logs[passage_logs['qid'] == qid]
    
    # 统计每种标签的文档数
    label_counts = query_passages['origin_label'].value_counts().to_dict()
    
    has_a_only = label_counts.get('A-Only', 0) > 0
    has_b_only = label_counts.get('B-Only', 0) > 0
    has_both = label_counts.get('Both', 0) > 0
    has_easy_neg = label_counts.get('Easy-Negative', 0) > 0
    has_unknown = label_counts.get('Unknown', 0) > 0
    
    # 统计唯一文档数
    a_only_docs = len(query_passages[query_passages['origin_label'] == 'A-Only']['docno'].unique())
    b_only_docs = len(query_passages[query_passages['origin_label'] == 'B-Only']['docno'].unique())
    
    # 判断是否可用于 ASR 分析
    valid_for_asr = has_a_only and has_b_only
    
    query_analysis.append({
        'qid': qid,
        'total_selections': len(query_passages),
        'has_a_only': has_a_only,
        'has_b_only': has_b_only,
        'has_both': has_both,
        'has_easy_neg': has_easy_neg,
        'has_unknown': has_unknown,
        'a_only_docs': a_only_docs,
        'b_only_docs': b_only_docs,
        'valid_for_asr': valid_for_asr,
        'a_only_selections': label_counts.get('A-Only', 0),
        'b_only_selections': label_counts.get('B-Only', 0),
        'both_selections': label_counts.get('Both', 0),
        'easy_neg_selections': label_counts.get('Easy-Negative', 0),
        'unknown_selections': label_counts.get('Unknown', 0)
    })

query_df = pd.DataFrame(query_analysis)

# ============================================================================
# 第六步：显示结果
# ============================================================================

print("Query Summary:")
print(f"  Total queries: {len(query_df)}")
print(f"  Valid for ASR (has both A-Only and B-Only): {query_df['valid_for_asr'].sum()}")
print(f"  Invalid for ASR: {(~query_df['valid_for_asr']).sum()}")
print()

# 显示无效的查询
invalid_queries = query_df[~query_df['valid_for_asr']]

if len(invalid_queries) > 0:
    print("=" * 80)
    print(f"Queries NOT Valid for ASR Analysis ({len(invalid_queries)} queries)")
    print("=" * 80)
    print()
    
    for idx, row in invalid_queries.iterrows():
        print(f"Query {row['qid']}:")
        print(f"  Total selections: {row['total_selections']}")
        print(f"  Has A-Only: {row['has_a_only']} ({row['a_only_docs']} docs, {row['a_only_selections']} selections)")
        print(f"  Has B-Only: {row['has_b_only']} ({row['b_only_docs']} docs, {row['b_only_selections']} selections)")
        print(f"  Has Both: {row['has_both']} ({row['both_selections']} selections)")
        print(f"  Has Easy-Negative: {row['has_easy_neg']} ({row['easy_neg_selections']} selections)")
        print(f"  Has Unknown: {row['has_unknown']} ({row['unknown_selections']} selections)")
        
        # 判断原因
        if not row['has_a_only'] and not row['has_b_only']:
            print("  → Reason: No A-Only or B-Only documents")
        elif not row['has_a_only']:
            print("  → Reason: No A-Only documents (cannot establish baseline)")
        elif not row['has_b_only']:
            print("  → Reason: No B-Only documents (nothing to compare)")
        
        print()

Diagnosing Missing Queries in ASR Analysis

Parsing SQL file: backup_qe_first_half.sql
✓ Logs parsed from SQL: 27466 records
✓ Interleaving results loaded: 387 records

Filtering for target users...
✓ Filtered logs: 7082 records from 27 users

✓ Time filtered: removed 486 old records

Document label mapping created: 387 documents

Total PASSAGE_SELECTION events: 2542

Query Analysis

Total unique queries in logs: 22

Query Summary:
  Total queries: 22
  Valid for ASR (has both A-Only and B-Only): 20
  Invalid for ASR: 2

Queries NOT Valid for ASR Analysis (2 queries)

Query 1121402:
  Total selections: 155
  Has A-Only: False (0 docs, 0 selections)
  Has B-Only: False (0 docs, 0 selections)
  Has Both: True (151 selections)
  Has Easy-Negative: True (4 selections)
  Has Unknown: False (0 selections)
  → Reason: No A-Only or B-Only documents

Query 1121709:
  Total selections: 46
  Has A-Only: False (0 docs, 0 selections)
  Has B-Only: False (0 docs, 0 selections)
  Has Both: True (46 s